554 Project 3

by Joshua McClure

Fitting Your Model (50 pts)

In [3]:
# Library for Packages
import pandas as pd
import time
import os
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

Part 1: Read in and Organize Data

Create a Jupyter notebook for the modeling fitting part and the Streaming part below.

* The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/ power_ml_data.csv

* You should read this data into a standard pandas data frame using the pd.read_csv() function.

* Convert this to a spark data frame

* We are going to treat the Power_Zone_3 variable as our response variable.

* We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [5]:
# Initialize SparkSession
spark = SparkSession.builder.appName("PowerDataProcessing").getOrCreate()

# Define the URL for the dataset
data_url = "https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv"

# Read the data into a pandas DataFrame, using the first row as headers
pd_df = pd.read_csv(data_url, header=0)

# Define the Spark schema based on the provided headers
spark_schema = StructType([
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("Wind_Speed", DoubleType(), True),
    StructField("General_Diffuse_Flows", DoubleType(), True),
    StructField("Diffuse_Flows", DoubleType(), True),
    StructField("Power_Zone_1", DoubleType(), True),
    StructField("Power_Zone_2", DoubleType(), True),
    StructField("Power_Zone_3", DoubleType(), True),
    StructField("Month", IntegerType(), True),
    StructField("Hour", IntegerType(), True)
])

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(pd_df, schema=spark_schema)

# Rename Power_Zone_3 to 'label' as it is our response variable
spark_df = spark_df.withColumnRenamed("Power_Zone_3", "label")

# Display schema and first few rows to verify
print("Spark DataFrame Schema:")
spark_df.printSchema()

print("\nFirst 5 rows of Spark DataFrame:")
spark_df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 12:53:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 12:53:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 12:53:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/30 12:53:13 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/30 12:53:13 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark DataFrame Schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- label: double (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)


First 5 rows of Spark DataFrame:


+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    1|   0|
+-----------+--------+----------+---------------

Part 2: Elastic Model Initial Pipeline Set Up

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read
in) with the steps below.

The transformations below should each use an MLlib function that can be put into a pipeline

  * The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the variable as a DoubleType

  * Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

  * One-hot encode the Month column

  * Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns.

    * To do this, I first used a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.

    * Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.

    * We’ll use two PCs in our transformation.

* Rename your response variable as label

* Use VectorAssembler() to put your predictors into a features. Use the:

  * two fitted PCA features

  * binary Hour variable

  * Power_Zone_1

  * Power_Zone_2

  * Month indicator variables

* This ends the pipeline of transformations!

In [4]:
# Step 1: Cast Hour column to DoubleType if not already
sql_transformer_hour = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double FROM __THIS__")

# Step 2: Binarize the Hour column (Night vs Day)
binarizer = Binarizer(threshold=6.5, inputCol="Hour_Double", outputCol="Hour_Binary")

# Step 3: One-hot encode the Month column
month_indexer = StringIndexer(inputCol="Month", outputCol="Month_Indexed")

# Then, OneHotEncoder to convert the indexed column to one-hot vectors
month_encoder = OneHotEncoder(inputCols=["Month_Indexed"], outputCols=["Month_OneHot"])

# Step 4: PCA on Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows
pca_input_cols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]
vector_assembler_pca = VectorAssembler(inputCols=pca_input_cols, outputCol="pca_features")

# Apply PCA to reduce dimensions to 2 principal components
pca = PCA(k=2, inputCol="pca_features", outputCol="principal_components")

# Step 5: Final VectorAssembler to put all predictors into a 'features' vector
# Use the two fitted PCA features, binary Hour, Power_Zone_1, Power_Zone_2, and Month indicator variables
final_features_assembler = VectorAssembler(
    inputCols=[
        "principal_components",
        "Hour_Binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_OneHot"
    ],
    outputCol="features"
)

# Step 6: Define the Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol="label", predictionCol="prediction")

# Create the pipeline
pipeline = Pipeline(stages=[
    sql_transformer_hour,
    binarizer,
    month_indexer,
    month_encoder,
    vector_assembler_pca,
    pca,
    final_features_assembler,
    lr # Add the Linear Regression model as the final stage of the pipeline
])

print("Pipeline stages defined successfully.")

Pipeline stages defined successfully.


Part 3: Fitting & Printing the Model

* Now you’ll then use the CrossValidator() function and the LinearRegression() function to fit an elastic net model.

  * You should do the following grid for the regParam and elasticNetParam: All combinations of

    * regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

    * elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

* Now fit the model using 5-fold CV with rmse as your criterion!

* Report the optimal values chosen for the tuning parameters

* Report the CV error

* Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and evaluating on the entire training set

* Take the outputted transformations from the model (the predictions) and create a residual column (label - prediction). The .withColumn() method is handy here. Print out a data frame with these residuals, the label column, and the predictions

In [5]:
# The Linear Regression model is already part of the pipeline defined in the previous cell.
# We need to retrieve that instance to correctly define the ParamGridBuilder.
lr_from_pipeline = None
for stage in pipeline.getStages():
    if isinstance(stage, LinearRegression):
        lr_from_pipeline = stage
        break

if lr_from_pipeline is None:
    raise ValueError("LinearRegression stage not found in the pipeline. Please ensure the pipeline in the previous cell is correctly defined and executed with a LinearRegression model.")

# Define the parameter grid for regParam and elasticNetParam
paramGrid = ParamGridBuilder() \
    .addGrid(lr_from_pipeline.regParam, [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1.0]) \
    .addGrid(lr_from_pipeline.elasticNetParam, [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1.0]) \
    .build()

# Create a RegressionEvaluator for RMSE
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

# Create the CrossValidator
cv = CrossValidator(
    estimator=pipeline, # Use the previously defined pipeline as the estimator
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5, 
    seed=42 
)

print("Fitting the model using CrossValidator...")
# Fit the model to the spark_df
cvModel = cv.fit(spark_df)

print("Model fitting complete.")

# --- Reporting Results ---

# The best model from CrossValidator is stored in cvModel.bestModel
best_pipeline_model = cvModel.bestModel

# Find the LinearRegressionModel stage in the best pipeline model
best_lr_model = None
for stage in best_pipeline_model.stages:
    if isinstance(stage, LinearRegression):
        best_lr_model = stage
        break

if best_lr_model:
    print(f"\nOptimal regParam: {best_lr_model.getRegParam()}")
    print(f"Optimal elasticNetParam: {best_lr_model.getElasticNetParam()}")
else:
    print("Could not find LinearRegression stage in the best model.")

# Report the CV error (average RMSE from cross-validation)
best_rmse = min(cvModel.avgMetrics)
print(f"\nCross-validation RMSE (best model): {best_rmse}")

# Report the training set RMSE
transformed_df = cvModel.transform(spark_df)

training_rmse = evaluator.evaluate(transformed_df)
print(f"Training set RMSE: {training_rmse}")

# Take the outputted transformations (predictions) and create a residual column
residuals_df = transformed_df.withColumn("residual", col("label") - col("prediction"))

print("\nDataFrame with Label, Prediction, and Residuals:")
residuals_df.select("label", "prediction", "residual").show(10)

Fitting the model using CrossValidator...


26/04/30 12:29:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 12:29:26 WARN Instrumentation: [6d27cfb4] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:29 WARN Instrumentation: [0289fe96] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:31 WARN Instrumentation: [673e76de] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:33 WARN Instrumentation: [ea633f71] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:34 WARN Instrumentation: [ff262512] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:35 WARN Instrumentation: [504962c7] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:29:36 WARN Instrumentation: [2d4795d1] regP

Model fitting complete.
Could not find LinearRegression stage in the best model.

Cross-validation RMSE (best model): 2147.5891325226876
Training set RMSE: 2147.097322400667

DataFrame with Label, Prediction, and Residuals:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20879.293929772837|-638.3300697728373|
|20131.08434|18659.581224684986|1471.5031153150157|
|19668.43373|18204.118378530166|1464.3153514698352|
|18899.27711|17590.065188636432| 1309.211921363567|
|18442.40964|16996.736644024797|1445.6729959752047|
|18130.12048| 16517.14980719943| 1612.970672800573|
|17945.06024|16092.738824696906| 1852.321415303093|
|17459.27711|15722.205354351358|1737.0717556486416|
|17025.54217| 15270.58168727468|1754.9604827253206|
|16794.21687|14937.899896525745| 1856.316973474255|
+-----------+------------------+------------------+
only showing top 10 rows


Streaming Part (40 pts)

There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
Download this file and store it where your .py file you’ll create can find it. We’ll be randomly sampling rows
from this to output to .csv files that you’ll be reading in.

In [6]:
# Define the schema for the streaming data
spark_schema_streaming = StructType([
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("Wind_Speed", DoubleType(), True),
    StructField("General_Diffuse_Flows", DoubleType(), True),
    StructField("Diffuse_Flows", DoubleType(), True),
    StructField("Power_Zone_1", DoubleType(), True),
    StructField("Power_Zone_2", DoubleType(), True),
    StructField("Power_Zone_3", DoubleType(), True),
    StructField("Month", IntegerType(), True),
    StructField("Hour", IntegerType(), True)
])

# Define the path to the streaming data file
streaming_data_path = "Power_Storage/power_streaming_data.csv"

# Read the data into a Spark DataFrame
streaming_df = spark.read.csv(
    streaming_data_path,
    header=True,
    schema=spark_schema_streaming
)

print("Streaming DataFrame Schema:")
streaming_df.printSchema()

print("First 5 rows of Streaming DataFrame:")
streaming_df.show(5)

Streaming DataFrame Schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)

First 5 rows of Streaming DataFrame:
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      4.805|    76.2|     0.081|                0.059|        0.134| 20421.26582| 12908.20669| 14590.84337|    1|   3|
|      4.212|    78

Part 1: Reading a Stream

* We’re going to read in a stream in the form of .csv files. Create a folder where you will be sending your .csv files.

* Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)

* Set up the readStream. Be sure to add header = True as you’ll likely be outputting files with a header and we don’t need to read that in.

In [7]:
# 1. Create a folder where you will be sending your .csv files
#    This will be the input directory for our Spark Structured Stream
streaming_input_dir = "Power_Storage/stream_data_input"
if not os.path.exists(streaming_input_dir):
    os.makedirs(streaming_input_dir)
    print(f"Created streaming input directory: {streaming_input_dir}")
else:
    print(f"Streaming input directory already exists: {streaming_input_dir}")

# 2. Set up the readStream
# We'll monitor the streaming_input_dir for new CSV files
stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(spark_schema_streaming) \
    .load(streaming_input_dir)

print("Spark Structured Stream configured to read from:")
print(f"  Directory: {streaming_input_dir}")
print(f"  Schema: {stream_df.printSchema()}")
print("Stream initialized successfully.")

Created streaming input directory: Power_Storage/stream_data_input
Spark Structured Stream configured to read from:
  Directory: Power_Storage/stream_data_input
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)

  Schema: None
Stream initialized successfully.
